In [1]:
!pip install optuna
%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import optuna 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader 
from sklearn.model_selection import train_test_split
from DataEncoder import encode_pad_event, encode_pad_sequence, scale_time_differences, encode_label_event, node_time_list, event_transition_edge, scale_time_differences_fast_fixed, length_stratified_split
from GATConvStatusEmb import prepare_data_core_2edges, prepare_data_y, CustomDataset, EarlyStopping, train, evaluate, custom_collate_fn, DualGAT2EdgesModel
from GATConvStatusEmb import predict, top_k_accuracy, predict_per_sequence, average_bleu_score, compute_dls_and_exact_match, sequence_level_top_k_accuracy, analyze_sequence_errors, predict_per_sequence_with_probs, sequence_level_top_k_analysis, show_error_sequences
import os 
import shutil

/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: dlopen(/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/libpyg.so, 0x0006): Library not loaded: /Library/Frameworks/Python.framework/Versions/3.11/Python
  Referenced from: <75FFC412-93B5-322B-8E6D-268DA3498CF4> /opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/libpyg.so
  Reason: tried: '/Library/Frameworks/Python.framework/Versions/3.11/Python' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/Python.framework/Versions/3.11/Python' (no such file), '/Library/Frameworks/Python.framework/Versions/3.11/Python' (no 

In [2]:
#event = pd.read_csv("../output/BPI12.csv")
# event = pd.read_csv("../output/BPI12w.csv")
#event = pd.read_csv("../output/BPI13i.csv")
#event = pd.read_csv("../output/BPI13c.csv")
event = pd.read_csv("../output/helpdesk.csv")

In [3]:
# check the size of sequence
shortest_sequence = event.groupby('sequence').size().min()
longest_sequence = event.groupby('sequence').size().max()
print('shortest_sequence:', shortest_sequence)
print('longest_sequence:', longest_sequence)
event = event[event.groupby('sequence')['sequence'].transform('size') >= 2].reset_index(drop=True)

shortest_sequence: 2
longest_sequence: 15


In [4]:
#core_event = 'event_label'
core_event = 'event' #substatus
case_index = 'sequence'

In [5]:
core_encode, y_encode, core_size, output_size, le_event = encode_label_event(event, core_event, case_index)

In [6]:
#event['ec1'] = event['ec1'].astype(str) 
# cat_col_event = ['ec1']
# num_col_event = []
#Bpi13
#cat_col_event = ['ec1', 'ec2', 'ec4', 'ec5']
#num_col_seq = []
#helpdesk
cat_col_event = ['ec1']
num_col_event = []
event_encode = encode_pad_event(event, cat_col_event, num_col_event, case_index, cat_mask = True, num_mask = True, eos = False)

In [7]:
# sequence = event[['sequence','sn1']].groupby('sequence').first()
#BPI13
#sequence = event[['sequence','sc3', 'sc1', 'sc2']].groupby('sequence').first()
sequence = event[['sequence','sc1']].groupby('sequence').first()
sequence = sequence.reset_index()
#sequence = event.groupby('sequence').apply(lambda x: x.iloc[prefix_size - 1:]).reset_index(drop=True)

In [8]:
# cat_col_seq = []
# num_col_seq = ['sn1']
#BPI13
#cat_col_seq = ['sc1','sc2','sc3']
#num_col_seq = []
#helpdesk
cat_col_seq = ['sc1']
num_col_seq = []
sequence_encode = encode_pad_sequence(sequence, cat_col_seq, num_col_seq)

In [9]:
status = 'event'
event_trans_edge, le_edge, trans_size = event_transition_edge(event, sequence, status, case_index)

In [10]:
start_time_col = 'time'
#start_time_col = 'Start Timestamp'
scaled_time_diffs = scale_time_differences_fast_fixed(event, sequence, start_time_col, case_index)

In [11]:
max_num_events = event_encode.shape[1]
# Expand sequence features to match the shape of event features
sequence_features_expanded = np.expand_dims(sequence_encode, axis=1)
sequence_features_expanded = np.repeat(sequence_features_expanded, max_num_events, axis=1)

# Combine event and sequence features
combined_features = np.concatenate((event_encode, sequence_features_expanded), axis=2)

In [12]:
num_sequences = event_encode.shape[0]
num_event_features = combined_features.shape[2]
num_embedding_features = core_size

In [13]:
event_feature_list = prepare_data_core_2edges(combined_features, core_encode, scaled_time_diffs, event_trans_edge)
y_list = prepare_data_y(combined_features, y_encode)

In [14]:
# Use stratified sampling over sequence length to preserve distributional characteristics
train_indices, test_indices = length_stratified_split(event_feature_list, test_size=0.2, n_bins=10)

# Split the data
train_event_features = [event_feature_list[i] for i in train_indices]
test_event_features = [event_feature_list[i] for i in test_indices]
train_y = [y_list[i] for i in train_indices]
test_y = [y_list[i] for i in test_indices]

# Print statistics
train_lengths = [event_feature_list[i].x.shape[0] for i in train_indices]
test_lengths = [event_feature_list[i].x.shape[0] for i in test_indices]

print(f"Train set: {len(train_indices)} samples")
print(f"Train length range: {min(train_lengths)} - {max(train_lengths)}")
print(f"Train length mean: {np.mean(train_lengths):.2f}")

print(f"\nTest set: {len(test_indices)} samples") 
print(f"Test length range: {min(test_lengths)} - {max(test_lengths)}")
print(f"Test length mean: {np.mean(test_lengths):.2f}")

Train set: 3666 samples
Train length range: 2 - 14
Train length mean: 4.57

Test set: 914 samples
Test length range: 3 - 15
Test length mean: 4.88


In [15]:
train_dataset = CustomDataset(train_event_features, train_y)
test_dataset = CustomDataset(test_event_features, test_y)

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# parameters
embedding_dims = 64
gat_hidden_dim_event = 32
gat_hidden_dim_embed = 128
gat_hidden_dim_concat = 256
output_dim = output_size  # vocab size
num_heads = 4
num_edge_types = trans_size
edge_type_dim = 32

model = DualGAT2EdgesModel(
    num_event_features=num_event_features,
    num_embedding_features=num_embedding_features,
    embedding_dims=embedding_dims,
    gat_hidden_dim_event=gat_hidden_dim_event,
    gat_hidden_dim_embed=gat_hidden_dim_embed,
    gat_hidden_dim_concat=gat_hidden_dim_concat,
    output_dim=output_dim,
    num_heads=num_heads,
    num_edge_types=num_edge_types,
    edge_type_dim=edge_type_dim
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=-1)  # ignore padding

In [17]:
batch_size = 16  
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

In [18]:
early_stopping = EarlyStopping(patience=3, delta=0.0)
num_epochs = 10

# Filepath to save model
model_path = "../output/models/BPI12w_transedge_es.pt"

config = {
    'model_type': 'DualGAT2EdgesModel',
    'num_event_features': num_event_features,
    'num_embedding_features': num_embedding_features,
    'embedding_dims': embedding_dims,
    'gat_hidden_dim_event': gat_hidden_dim_event,
    'gat_hidden_dim_embed': gat_hidden_dim_embed,
    'gat_hidden_dim_concat': gat_hidden_dim_concat,
    'output_dim': output_dim,
    'num_heads': num_heads,
    'num_edge_types': num_edge_types,
    'edge_type_dim': edge_type_dim
}

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
    
    if early_stopping(test_loss):
        print("Early stopping triggered.")
        break

    if early_stopping.best_loss_updated:
        print(f"New best model at epoch {epoch+1}, saving to {model_path}")
        best_model_saved = True
         # Save state dict and config
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)
        
if not early_stopping.early_stop and not best_model_saved:
        print("Training completed without early stopping. Saving final model.")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)

Epoch 1/10 | Train Loss: 0.3813 Acc: 0.8870 | Test Loss: 0.3907 Acc: 0.8749
New best model at epoch 1, saving to ../output/models/BPI12w_transedge_es.pt
Epoch 2/10 | Train Loss: 0.2129 Acc: 0.9280 | Test Loss: 0.2888 Acc: 0.9168
New best model at epoch 2, saving to ../output/models/BPI12w_transedge_es.pt
Epoch 3/10 | Train Loss: 0.1724 Acc: 0.9363 | Test Loss: 0.2721 Acc: 0.9222
New best model at epoch 3, saving to ../output/models/BPI12w_transedge_es.pt
Epoch 4/10 | Train Loss: 0.1492 Acc: 0.9429 | Test Loss: 0.2712 Acc: 0.9215
New best model at epoch 4, saving to ../output/models/BPI12w_transedge_es.pt
Epoch 5/10 | Train Loss: 0.1398 Acc: 0.9464 | Test Loss: 0.2924 Acc: 0.9224
Epoch 6/10 | Train Loss: 0.1314 Acc: 0.9470 | Test Loss: 0.3076 Acc: 0.9190
Epoch 7/10 | Train Loss: 0.1296 Acc: 0.9498 | Test Loss: 0.3188 Acc: 0.9204
Early stopping triggered.


In [19]:
preds, labels, outs = predict(model, test_loader, device)

# Check some predictions
for i in range(10):
    print(f"Predicted: {preds[i].item()} | Actual: {labels[i].item()}")

Predicted: 10 | Actual: 10
Predicted: 1 | Actual: 1
Predicted: 5 | Actual: 5
Predicted: 10 | Actual: 10
Predicted: 1 | Actual: 1
Predicted: 5 | Actual: 5
Predicted: 10 | Actual: 10
Predicted: 1 | Actual: 1
Predicted: 5 | Actual: 5
Predicted: 10 | Actual: 10


In [20]:
top1 = top_k_accuracy(outs, labels, k=1)
top3 = top_k_accuracy(outs, labels, k=3)
top5 = top_k_accuracy(outs, labels, k=5)

print(f"Top-1 Acc: {top1:.4f} | Top-3 Acc: {top3:.4f} | Top-5 Acc: {top5:.4f}")

Top-1 Acc: 0.9204 | Top-3 Acc: 0.9832 | Top-5 Acc: 0.9913


In [21]:
preds_seq, labels_seq = predict_per_sequence(model, test_loader, device)

In [22]:
bleu = average_bleu_score(preds_seq, labels_seq)
print(f"Average BLEU Score: {bleu:.4f}")

Average BLEU Score: 0.8453


In [23]:
avg_dls, exact_acc = compute_dls_and_exact_match(preds_seq, labels_seq)
print(f"Damerau-Levenshtein Similarity (avg): {avg_dls:.4f}")
print(f"Exact Match Accuracy: {exact_acc:.4f}")

Damerau-Levenshtein Similarity (avg): 0.9411
Exact Match Accuracy: 0.7757


In [24]:
# Usage
seq_top3_acc = sequence_level_top_k_accuracy(model, test_loader, device, k=3)
print(f"Sequence-Level Top-3 Accuracy: {seq_top3_acc:.4f}")
seq_top5_acc = sequence_level_top_k_accuracy(model, test_loader, device, k=5)
print(f"Sequence-Level Top-5 Accuracy: {seq_top5_acc:.4f}")

Sequence-Level Top-3 Accuracy: 0.9464
Sequence-Level Top-5 Accuracy: 0.9694


In [25]:
# Usage example
pos_errors, error_types, seq_stats = analyze_sequence_errors(model, test_loader, device, k=3)


=== Error Analysis Report ===
Total sequences analyzed: 914
Total sequences with errors: 205
Sequence length stats: {'min': 3, 'max': 15, 'mean': 4.878555798687089, 'median': 4.0}

Error distribution by position:
Position 0: 37 errors (10.4% of all errors)
Position 1: 77 errors (21.7% of all errors)
Position 2: 80 errors (22.5% of all errors)
Position 3: 106 errors (29.9% of all errors)
Position 4: 21 errors (5.9% of all errors)
Position 5: 15 errors (4.2% of all errors)
Position 6: 8 errors (2.3% of all errors)
Position 7: 5 errors (1.4% of all errors)
Position 8: 2 errors (0.6% of all errors)
Position 9: 2 errors (0.6% of all errors)
Position 10: 1 errors (0.3% of all errors)
Position 11: 1 errors (0.3% of all errors)

Most common error types:
Predicted 1 instead of 10: 51 times
Predicted 10 instead of 14: 47 times
Predicted 12 instead of 0: 40 times
Predicted 10 instead of 12: 34 times
Predicted 14 instead of 10: 29 times
Predicted 14 instead of 12: 22 times
Predicted 0 instead of 

In [26]:
# First get predictions with probabilities
all_preds, all_labels, all_topk = predict_per_sequence_with_probs(model, test_loader, device, k=3)

# Then analyze sequence-level top-k accuracy
topk_acc, error_stats = sequence_level_top_k_analysis(all_topk, all_labels)

print(f"Sequence-Level Top-3 Accuracy: {topk_acc:.4f}")
print("\nError Analysis:")
print(f"- Most error-prone positions: {error_stats['position_errors']}")
print(f"- Top prediction mistakes: {error_stats['top_errors']}")

Sequence-Level Top-3 Accuracy: 0.9464

Error Analysis:
- Most error-prone positions: {0: 2, 1: 13, 2: 17, 3: 16, 4: 10, 5: 10, 6: 4, 7: 2, 8: 0, 9: 1}
- Top prediction mistakes: {(14, 12): 11, (10, 12): 9, (14, 10): 8, (1, 10): 7, (1, 12): 6}


In [27]:
show_error_sequences(all_topk, all_labels, num=3)


Error Sequence #19:
True: [12, 10, 5]
Pred: [12, 14, 12]
Mismatches:
Pos 1: True 10 not in [14, 8, 2]

Error Sequence #556:
True: [0, 0, 10, 10, 1, 5]
Pred: [12, 12, 12, 12, 1, 5]
Mismatches:
Pos 2: True 10 not in [12, 0, 14]
Pos 3: True 10 not in [12, 0, 1]

Error Sequence #566:
True: [12, 2, 8, 8, 10, 5]
Pred: [12, 14, 10, 10, 10, 1]
Mismatches:
Pos 3: True 8 not in [10, 13, 14]
Pos 5: True 5 not in [1, 7, 10]


In [28]:
from sklearn.metrics import classification_report
unique_true = np.unique(labels.cpu().numpy())
unique_preds = np.unique(preds.cpu().numpy())

print("Classes in true labels:", unique_true)
print("Classes in predictions:", unique_preds)
print("Missing classes:", set(unique_true) - set(unique_preds))

actual_classes = np.union1d(unique_true, unique_preds)
report = classification_report(
    labels.cpu().numpy(),
    preds.cpu().numpy(),
    target_names=[f"Class_{i}" for i in actual_classes] ,
    digits=4
)
print(report)

Classes in true labels: [ 0  1  2  5  8  9 10 11 12 14]
Classes in predictions: [ 0  1  2  5  7  8  9 10 12 13 14]
Missing classes: {11}
              precision    recall  f1-score   support

     Class_0     0.6324    0.5181    0.5695        83
     Class_1     0.9184    0.9923    0.9539       907
     Class_2     0.7500    0.1875    0.3000        16
     Class_5     0.9978    0.9923    0.9951       914
     Class_7     0.0000    0.0000    0.0000         0
     Class_8     0.8571    0.3636    0.5106        33
     Class_9     0.0000    0.0000    0.0000         4
    Class_10     0.9000    0.8991    0.8996      1001
    Class_11     0.0000    0.0000    0.0000         2
    Class_12     0.9403    0.9206    0.9303      1146
    Class_13     0.0000    0.0000    0.0000         0
    Class_14     0.7955    0.8045    0.8000       353

    accuracy                         0.9204      4459
   macro avg     0.5660    0.4732    0.4966      4459
weighted avg     0.9188    0.9204    0.9178      44

/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/gnn-pbpm/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.s

In [29]:
# Load model state
checkpoint = torch.load(model_path, map_location=device)
config = checkpoint['config']

# Rebuild the model using the same config
model = DualGAT2EdgesModel(
    num_event_features=config['num_event_features'],
    num_embedding_features=config['num_embedding_features'],
    embedding_dims=config['embedding_dims'],
    gat_hidden_dim_event=config['gat_hidden_dim_event'],
    gat_hidden_dim_embed=config['gat_hidden_dim_embed'],
    gat_hidden_dim_concat=config['gat_hidden_dim_concat'],
    output_dim=config['output_dim'],
    num_heads=config['num_heads'],
    num_edge_types=config['num_edge_types'],
    edge_type_dim=config['edge_type_dim']
)

model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

# Rebuild optimizer (if needed)
optimizer = torch.optim.Adam(model.parameters())
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Recover attention and event IDs
start_epoch = checkpoint['epoch'] + 1 

for epoch in range(start_epoch, 10):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")
    
    if early_stopping(test_loss):
        print("Early stopping triggered.")
        break

    if early_stopping.best_loss_updated:
        print(f"New best model at epoch {epoch+1}, saving to {model_path}")
        best_model_saved = True
         # Save state dict and config
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)
        
if not early_stopping.early_stop and not best_model_saved:
        print("Training completed without early stopping. Saving final model.")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
            'config': config
        }, model_path)

Epoch 5/10 | Train Loss: 0.1389 Acc: 0.9453 | Test Loss: 0.2736 Acc: 0.9267
Early stopping triggered.
